# Full-variable autoregressive ablation

This mirrors TEP notebook 4.0. It predicts each endogenous process variable from the lagged history of all six variables and scores only the three endogenous-variable residuals with their full training residual precision matrix. The moving-average window and consecutive-alarm count are fixed to six and five, respectively; the threshold is the maximum healthy-phase score across faulty validation runs. Test runs are not used to fit models or select the threshold.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display
from carl.causalai_baselines import run_ablation_comparison

ablation_results = run_ablation_comparison("full_var", PROJECT_ROOT)
display(ablation_results["selected_settings"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold
0,full_var,mahalanobis,2,6,5,3.4299


# 2. Threshold Selection

The threshold is the maximum six-sample moving-average score in the pre-fault portions of the faulty validation trajectories, matching TEP's ablation procedure.

In [2]:
display(ablation_results["validation_metrics"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,full_var,mahalanobis,2,6,5,3.4299,0.492763,0.372928,0.0,0.543259,0.895556,0.791111,0.0,0.883375,178,225,32.5


# 3. Test on Faulty Time Series

Evaluate the selected ablation settings on every faulty held-out test run.

In [3]:
display(ablation_results["faulty_test_metrics"])
display(ablation_results["faulty_test_by_type"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,full_var,mahalanobis,2,6,5,3.4299,0.511075,0.395567,0.0,0.56689,0.902222,0.804444,0.0,0.891626,181,225,31.0


,Fault Type,Runs,Detected_Runs,Run_Detection_Rate,Median_Detection_Delay
0,hydraulic_coupling_shift,75,35,0.466667,256.0
1,reaction_coupling_shift,75,75,1.000000,19.0
2,thermal_coupling_shift,75,71,0.946667,38.0


# 4. Test on Fault-Free Time Series

Evaluate the same fixed settings on all healthy held-out test runs.

In [4]:
display(ablation_results["healthy_test_metrics"])
display(ablation_results["comparison"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase FPR,False Alarm Runs,Total Healthy Runs
0,full_var,mahalanobis,2,6,5,3.4299,1.0,0.0,0.0,0.0,0.0,0,75


,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Faulty Test Sample F1,Faulty Test Trajectory-Phase F1,Faulty Test Trajectory-Phase TPR,Faulty Test Median Detection Delay,Healthy Test Sample FPR,Healthy Test Trajectory-Phase FPR
0,diagonal_covariance,diagonal,2,6,5,3.429457,0.566845,0.891626,0.804444,31.0,0.0,0.0
4,full_var,mahalanobis,2,6,5,3.429900,0.566890,0.891626,0.804444,31.0,0.0,0.0
1,pca,spe,2,6,5,2.360392,0.362259,0.821990,0.697778,52.0,0.0,0.0
2,plain_residuals,plain,2,6,5,0.793246,0.536764,0.815789,0.688889,32.0,0.0,0.0
3,self_history,diagonal,2,6,5,3.495081,0.475545,0.721591,0.564444,32.0,0.0,0.0
